## Evaluación de métricas

In [ ]:
# ============================================================
# 9.10.4.6  COMPARACIÓN DE RESULTADOS sklearn vs PySpark
# ============================================================
from sklearn.metrics import roc_curve, auc as sk_auc

# ── Métricas sklearn (calculadas previamente) ────────────────────────────────
sk_metrics = {
    "ROC AUC"  : auc,
    "Accuracy" : accuracy,
    "Precision": precision,
    "Recall"   : recall,
    "F1-score" : f1
}

# ── Métricas PySpark (desde matriz de confusión manual) ──────────────────────
# Sustituye con tus valores reales de TP/TN/FP/FN de PySpark
TP_sp, TN_sp, FP_sp, FN_sp = 11, 214885, 4, 53794

auc_sp       = 0.6975           # BinaryClassificationEvaluator
acc_sp       = (TP_sp+TN_sp)/(TP_sp+TN_sp+FP_sp+FN_sp)
prec_sp      = TP_sp/(TP_sp+FP_sp) if (TP_sp+FP_sp)>0 else 0
rec_sp       = TP_sp/(TP_sp+FN_sp) if (TP_sp+FN_sp)>0 else 0
f1_sp        = 2*prec_sp*rec_sp/(prec_sp+rec_sp) if (prec_sp+rec_sp)>0 else 0

spark_metrics = {
    "ROC AUC"  : auc_sp,
    "Accuracy" : acc_sp,
    "Precision": prec_sp,
    "Recall"   : rec_sp,
    "F1-score" : f1_sp
}

# ── Tiempos (usa tus variables de tiempo reales) ─────────────────────────────
tiempos = {
    "Entrenamiento (CV)": [elapsed_train,       elapsed_train_spark],
    "Predicción"        : [elapsed_pred,         elapsed_pred_spark],
}

# ── Tabla comparativa ────────────────────────────────────────────────────────
import pandas as pd
tabla = pd.DataFrame({
    "scikit-learn": sk_metrics,
    "PySpark"     : spark_metrics
}).round(4)

print("\n══════════════════════════════════════════════════════")
print("       TABLA COMPARATIVA — sklearn vs PySpark")
print("══════════════════════════════════════════════════════")
print(tabla.to_string())

tabla_tiempos = pd.DataFrame(tiempos, index=["scikit-learn", "PySpark"]).T.round(2)
print("\n  Tiempos (segundos):")
print(tabla_tiempos.to_string())

# ── Curva ROC sklearn ────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc_val  = sk_auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC
axes[0].plot(fpr, tpr, color="#2980b9", lw=2,
             label=f"sklearn  (AUC = {roc_auc_val:.4f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
# PySpark ROC solo como punto de referencia (AUC escalar)
axes[0].axhline(auc_sp, color="#e74c3c", linestyle=":",
                label=f"PySpark  (AUC = {auc_sp:.4f})")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("Curva ROC — sklearn vs PySpark")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Comparación de tiempos
categorias = list(tiempos.keys())
x          = np.arange(len(categorias))
width      = 0.35
sk_times   = [tiempos[c][0] for c in categorias]
sp_times   = [tiempos[c][1] for c in categorias]

bars1 = axes[1].bar(x - width/2, sk_times, width, label="scikit-learn",
                    color="#2980b9", alpha=0.85)
bars2 = axes[1].bar(x + width/2, sp_times, width, label="PySpark",
                    color="#e74c3c", alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(categorias)
axes[1].set_ylabel("Segundos")
axes[1].set_title("Comparación de tiempos")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 f"{bar.get_height():.1f}s",
                 ha="center", va="bottom", fontsize=9)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 f"{bar.get_height():.1f}s",
                 ha="center", va="bottom", fontsize=9)

plt.suptitle("Comparación sklearn vs PySpark — RandomForestClassifier", fontsize=13)
plt.tight_layout()
plt.show()